# CARLA scenario analysis — cross-condition comparison

Loads the latest run for each condition (`none`, `raw`, `camouflaged`) from
`experiments/carla_scenarios/` and compares:
1. Per-condition overview (brake / throttle / steer / speeds / distance)
2. Cross-condition overlays per agent
3. Aggregate stats per phase (cruise / patch / brake) × agent × condition
4. Distance and collision comparison

Scenario phases: **cruise** [0-5s]  **patch** [5-10s]  **brake** [10-15s]

In [ ]:
import json
import os
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SCRIPT_DIR = os.path.abspath(os.getcwd())
REPO_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, '..', '..'))
RUNS_DIR = os.path.join(REPO_ROOT, 'experiments', 'carla_scenarios')

CONDITIONS = ('none', 'raw', 'camouflaged')
COND_COLORS = {'none': '#2ca02c', 'raw': '#d62728', 'camouflaged': '#ff7f0e'}

# Auto-pick latest run per condition
RUN_BY_COND = {}
for cond in CONDITIONS:
    cands = sorted(glob(os.path.join(RUNS_DIR, f'{cond}_*')), key=os.path.getmtime)
    cands = [c for c in cands if os.path.isdir(c)]
    if cands:
        RUN_BY_COND[cond] = cands[-1]

print('Runs selected:')
for c, p in RUN_BY_COND.items():
    print(f'  {c:12s} -> {os.path.basename(p)}')
print(f'Missing: {[c for c in CONDITIONS if c not in RUN_BY_COND]}')

In [ ]:
def load_run(run_dir):
    with open(os.path.join(run_dir, 'summary.json')) as f:
        summary = json.load(f)
    tele = pd.read_csv(os.path.join(run_dir, 'telemetry.csv'))
    agents = {}
    for p in sorted(glob(os.path.join(run_dir, 'agent_*.csv'))):
        name = os.path.basename(p)[len('agent_'):-len('.csv')]
        agents[name] = pd.read_csv(p)
    return {'summary': summary, 'tele': tele, 'agents': agents}

RUNS = {c: load_run(p) for c, p in RUN_BY_COND.items()}

# Phase boundaries from first available run
ref = next(iter(RUNS.values()))['summary']
TICK_DT = ref['sim_duration_s'] / ref['num_ticks']
CRUISE_END = ref['cruise_end_tick'] * TICK_DT
BRAKE_START = ref['brake_start_tick'] * TICK_DT
SIM_END = ref['sim_duration_s']
AGENT_NAMES = sorted({a for r in RUNS.values() for a in r['agents']})

print(f'Phases: cruise [0, {CRUISE_END}s]  patch [{CRUISE_END}, {BRAKE_START}s]  brake [{BRAKE_START}, {SIM_END}s]')
print(f'Agents: {AGENT_NAMES}')
for cond, r in RUNS.items():
    s = r['summary']
    print(f"  {cond:12s} collisions={s['total_collisions']:>3}  min_dist={s['min_distance_m']:.2f}m  mean_dist={s['mean_distance_m']:.2f}m")

## 1. Per-condition overview

One column per condition, stacked plots: brake / throttle / steer / speeds / distance.

In [ ]:
def shade_phases(ax):
    ax.axvspan(0, CRUISE_END, alpha=0.08, color='green')
    ax.axvspan(CRUISE_END, BRAKE_START, alpha=0.08, color='orange')
    ax.axvspan(BRAKE_START, SIM_END, alpha=0.12, color='red')

n_cond = len(RUNS)
fig, axes = plt.subplots(5, n_cond, figsize=(6 * n_cond, 14), sharex=True, squeeze=False)
for col, (cond, r) in enumerate(RUNS.items()):
    tele = r['tele']
    agents = r['agents']
    s = r['summary']
    axes[0, col].set_title(f"{cond}  -- collisions={s['total_collisions']}, min_dist={s['min_distance_m']:.2f}m",
                           fontweight='bold', color=COND_COLORS[cond])

    for name, df in agents.items():
        axes[0, col].plot(df['sim_time_s'], df['brake'], label=name, alpha=0.85)
        axes[1, col].plot(df['sim_time_s'], df['throttle'], label=name, alpha=0.85)
        axes[2, col].plot(df['sim_time_s'], df['steer'], label=name, alpha=0.85)

    axes[3, col].plot(tele['sim_time_s'], tele['leader_speed_kmh'], label='leader', lw=2, color='black')
    axes[3, col].plot(tele['sim_time_s'], tele['follower_speed_kmh'], label='follower', lw=2, color='tab:blue')
    axes[4, col].plot(tele['sim_time_s'], tele['distance_m'], color='purple', lw=2)

    for row in range(5):
        shade_phases(axes[row, col])
    axes[0, col].set_ylim(-0.05, 1.1)
    axes[1, col].set_ylim(-0.05, 1.1)
    axes[2, col].axhline(0, color='k', lw=0.5)
    axes[0, col].legend(loc='upper right', ncol=2, fontsize=8)
    axes[3, col].legend(loc='upper right', fontsize=8)
    axes[4, col].set_xlabel('sim time (s)')

for row, label in enumerate(['brake', 'throttle', 'steer', 'speed (km/h)', 'distance (m)']):
    axes[row, 0].set_ylabel(label, fontsize=11, fontweight='bold')

plt.tight_layout(); plt.show()

## 2. Cross-condition overlay per agent

For each agent, overlay the three conditions on the same axes. This is the main view:
if the patch has an effect, the `raw` / `camouflaged` curves will diverge from `none`.

In [ ]:
metrics = ['brake', 'throttle', 'steer']
fig, axes = plt.subplots(len(AGENT_NAMES), len(metrics),
                         figsize=(5 * len(metrics), 3.2 * len(AGENT_NAMES)), sharex=True)
if len(AGENT_NAMES) == 1:
    axes = axes[np.newaxis, :]

for row, agent in enumerate(AGENT_NAMES):
    for col, metric in enumerate(metrics):
        ax = axes[row, col]
        for cond, r in RUNS.items():
            if agent not in r['agents']:
                continue
            df = r['agents'][agent]
            ax.plot(df['sim_time_s'], df[metric], label=cond, color=COND_COLORS[cond], alpha=0.85)
        shade_phases(ax)
        if metric in ('brake', 'throttle'):
            ax.set_ylim(-0.05, 1.1)
        if metric == 'steer':
            ax.axhline(0, color='k', lw=0.5)
        if row == 0:
            ax.set_title(metric, fontweight='bold')
        if col == 0:
            ax.set_ylabel(agent, fontsize=9, fontweight='bold')
        if row == len(AGENT_NAMES) - 1:
            ax.set_xlabel('sim time (s)')
        if row == 0 and col == len(metrics) - 1:
            ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Agent control signals across conditions', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Aggregate stats per phase

Mean throttle, brake, |steer| computed inside each scenario phase, for every (agent, condition).
This is the main quantitative view — look at the `patch` phase for the direct effect of the attack.

In [ ]:
def phase_of(t):
    if t < CRUISE_END: return 'cruise'
    if t < BRAKE_START: return 'patch'
    return 'brake'

rows = []
for cond, r in RUNS.items():
    for agent, df in r['agents'].items():
        df = df.copy()
        df['phase'] = df['sim_time_s'].apply(phase_of)
        for phase, sub in df.groupby('phase'):
            rows.append({
                'condition': cond, 'agent': agent, 'phase': phase,
                'mean_throttle': sub['throttle'].mean(),
                'mean_brake': sub['brake'].mean(),
                'mean_abs_steer': sub['steer'].abs().mean(),
                'ticks_brake>0.5': int((sub['brake'] > 0.5).sum()),
                'ticks_throttle>0.1': int((sub['throttle'] > 0.1).sum()),
                'n_ticks': len(sub),
            })
stats = pd.DataFrame(rows)
stats['phase'] = pd.Categorical(stats['phase'], ['cruise', 'patch', 'brake'])
stats = stats.sort_values(['agent', 'phase', 'condition']).reset_index(drop=True)
stats.round(3)

In [ ]:
# Pivot: one table per metric -> agent x (phase, condition)
for metric in ['mean_throttle', 'mean_brake', 'mean_abs_steer']:
    pv = stats.pivot_table(index='agent', columns=['phase', 'condition'], values=metric)
    print(f'\n=== {metric} ===')
    print(pv.round(3).to_string())

In [ ]:
# Delta vs 'none' during the patch phase — most directly shows the attack effect
if 'none' in RUNS:
    patch_phase = stats[stats['phase'] == 'patch']
    pv = patch_phase.pivot_table(index='agent', columns='condition',
                                 values=['mean_brake', 'mean_throttle', 'mean_abs_steer'])
    deltas = {}
    for metric in ['mean_brake', 'mean_throttle', 'mean_abs_steer']:
        base = pv[metric].get('none')
        for cond in ('raw', 'camouflaged'):
            if cond in pv[metric].columns:
                deltas[f'd_{metric}[{cond}]'] = pv[metric][cond] - base
    delta_df = pd.DataFrame(deltas)
    print('Delta vs none during PATCH phase (positive = attack increased the signal):')
    print(delta_df.round(4).to_string())
else:
    print("'none' condition missing — cannot compute deltas")

## 4. Leader / follower speed and distance across conditions

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

for cond, r in RUNS.items():
    tele = r['tele']
    col = COND_COLORS[cond]
    axes[0].plot(tele['sim_time_s'], tele['follower_speed_kmh'], label=f'follower {cond}', color=col)
    axes[1].plot(tele['sim_time_s'], tele['distance_m'], label=cond, color=col)
    # TTC: -1 means no closure; mask those out
    ttc = tele['ttc_s'].where(tele['ttc_s'] > 0)
    axes[2].plot(tele['sim_time_s'], ttc, label=cond, color=col)

# Leader (same across conditions in principle) once
tele0 = next(iter(RUNS.values()))['tele']
axes[0].plot(tele0['sim_time_s'], tele0['leader_speed_kmh'], label='leader', color='black', lw=2, ls='--')

for ax, ylab in zip(axes, ['speed (km/h)', 'distance (m)', 'TTC (s)']):
    shade_phases(ax)
    ax.set_ylabel(ylab)
    ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('sim time (s)')
axes[2].set_ylim(0, 20)  # clip absurd TTCs
fig.suptitle('Ground-truth trajectory — speed, distance, TTC', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Collision summary

In [ ]:
coll_rows = []
for cond, r in RUNS.items():
    s = r['summary']
    coll_rows.append({
        'condition': cond,
        'collisions': s['total_collisions'],
        'min_dist_m': s['min_distance_m'],
        'mean_dist_m': s['mean_distance_m'],
    })
coll = pd.DataFrame(coll_rows).set_index('condition')

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
colors = [COND_COLORS[c] for c in coll.index]
axes[0].bar(coll.index, coll['collisions'], color=colors); axes[0].set_title('Total collisions')
axes[1].bar(coll.index, coll['min_dist_m'], color=colors); axes[1].set_title('Min distance (m)')
axes[2].bar(coll.index, coll['mean_dist_m'], color=colors); axes[2].set_title('Mean distance (m)')
plt.tight_layout(); plt.show()

coll.round(3)